# On-premises Agent Framework + confidential MCP tools

The **agent loop runs on-premises**, Microsoft Foundry provides model inference, and the tools submit calculations to the private confidential ACI worker. This does not deploy a hosted agent and does not use Foundry's built-in code interpreter.

Complete [notebook 1](01_mcp_tools.ipynb) first. Use the same kernel and `.env`, adding `FOUNDRY_PROJECT_ENDPOINT` and `FOUNDRY_MODEL`. The Foundry identity comes from a **separate `DefaultAzureCredential`**, for example your `az login`, with permission to invoke the project/model (normally Azure AI User on the appropriate Foundry scope). It is not the sandbox bearer token. See [authentication](../docs/getting-started.md#authentication-from-on-premises).

**Data boundary:** only synthetic inputs are used. Tool summaries are sent to the Foundry model, so those results leave the TEE. File bytes are uploaded/downloaded by trusted local code. Secrets, access tokens, and session IDs are not model-visible function arguments. `store=False` requests no response storage; it is not a guarantee about all service telemetry or retention.

The equivalent runnable script is [agent_framework_analysis.py](agent_framework_analysis.py). Running this notebook incurs model and worker charges on your configured services.

In [ ]:
import asyncio
import sys
from pathlib import Path

root = Path.cwd().resolve()
if root.name == 'examples':
    root = root.parent
if not (root / 'examples' / 'pool_client.py').is_file():
    raise RuntimeError('Start the kernel in the repository root or examples directory')
sys.path.insert(0, str(root / 'examples'))

from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from azure.ai.projects.aio import AIProjectClient
from azure.identity.aio import DefaultAzureCredential
from pool_client import EXAMPLES, PoolClient, PoolConfig, required_setting
from agent_framework_analysis import PROMPT, session_tools, validate_summary

config = PoolConfig.from_env()
endpoint = required_setting('FOUNDRY_PROJECT_ENDPOINT')
model = required_setting('FOUNDRY_MODEL')
print('MCP and Foundry configuration loaded without displaying credentials.')

## Bind tools to one session in trusted code

`session_tools(pool, session_id)` returns two small typed async functions: `run_python(code)` and `list_files()`. They call MCP `run_python` and `list_dir` respectively. The closure supplies the lease ID; the model cannot pick a different session or control allocation/deletion. Read the function definitions in [the script](agent_framework_analysis.py) to see the exact calls.

Only `input.xlsx` is uploaded. The agent creates `summary.csv`; local code downloads and verifies the actual CSV against the known synthetic totals, rather than accepting the model's success claim. Calls are sequential and bounded to eight iterations/eight functions and five minutes. A tool/lease error is not automatically replayed.

In [ ]:
async with DefaultAzureCredential() as foundry_credential:
    async with AIProjectClient(endpoint=endpoint, credential=foundry_credential) as project:
        client = FoundryChatClient(
            project_client=project,
            model=model,
            function_invocation_configuration={
                'max_iterations': 8,
                'max_function_calls': 8,
                'max_duration_seconds': 300,
                'allow_concurrent_invocation': False,
            },
        )
        async with PoolClient(config) as pool:
            async with pool.session() as session_id:
                await pool.upload(session_id, EXAMPLES / 'sample_sales.xlsx')
                async with Agent(
                    client=client,
                    name='confidential-spreadsheet-analyst',
                    instructions='Use tools for calculations. Only use synthetic data. Return small aggregates, not source rows.',
                    tools=session_tools(pool, session_id),
                    default_options={'store': False},
                ) as agent:
                    async with asyncio.timeout(300):
                        response = await agent.run(PROMPT)
                artifact = await pool.download(session_id, 'summary.csv', EXAMPLES / 'agent_summary.csv')
                print(validate_summary(artifact))
                print(response.text)
print('Verified synthetic summary; session released. Worker deletion is asynchronous.')

## Operational limits

The app identity owns the session. Reusing one service principal across many people does **not** add per-human isolation; a production front-end must define its user authorization model separately. Treat app credentials as application secrets, not credentials to distribute to every notebook user. For production daemons prefer an Entra certificate credential or approved federation; do not check private keys into this repository.

Network timeout: check on-premises-to-VNet routing, gateway private DNS, and outbound reachability to Entra and Foundry. A Foundry permission/model error is independent of MCP role assignment. A certificate error needs a trusted new fingerprint, not `verify=False`. Missing or incorrect CSV output fails explicitly even when the agent returns a fluent explanation.

Normal failures/cancellation release the session. A killed kernel or broken connection may require `list_sessions`/`delete_session` recovery as described in [notebook 1](01_mcp_tools.ipynb). Do not rerun unknown-outcome code automatically. Workers expire on the configured idle/absolute limits.

This notebook is shipped without outputs. Offline verification uses explicit fakes and does not establish live model, networking, attestation, or worker availability. Run the [live release gate](../docs/operations.md#live-release-gate) before claiming a working deployment.